# 05 · Temporal model (1D-CNN + BiLSTM)

Same features, but as 6.4 s sequences instead of per-window statistics, so the
model can use the *order* of events.

Only run this after notebook 04: the baseline is the reference the temporal
model has to beat.

## Shared setup (run first)

This notebook uses the modules in `ml/`. Upload them once to the Colab runtime:

1. Click the folder icon in the left sidebar (Files).
2. Click **Upload to session storage** and select `extract_pose.py`,
   `fitness_features.py`, `build_dataset.py`, `train_baseline.py`,
   `train_temporal.py`, `export_onnx.py` from `F:\Fitness Coach\ml`.
3. Run the cell below.

The dataset zip is expected at
`/content/drive/MyDrive/Dataset Exercise Quality-wise.zip` (Google Drive).

In [ ]:
import os, sys, subprocess, json, glob
from pathlib import Path

ROOT = Path('/content/fitness')
ROOT.mkdir(parents=True, exist_ok=True)

# The uploaded modules land in /content; mirror them into the work root.
for f in ['extract_pose.py', 'fitness_features.py', 'build_dataset.py', 'train_baseline.py',
          'experiment_generalization.py', 'experiment_clip.py', 'experiment_repmetry.py',
          'train_final.py', 'train_temporal.py', 'export_onnx.py']:
    src = Path('/content') / f
    if src.exists():
        dst = ROOT / f
        if not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime:
            dst.write_bytes(src.read_bytes())

sys.path.insert(0, str(ROOT))
print('work root :', ROOT)
print('modules   :', sorted(p.name for p in ROOT.glob('*.py')))
print('cuda      :', subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True).stdout.strip() or 'no GPU')

In [ ]:
!cd /content/fitness && python -u /content/fitness/train_temporal.py

## Compare against the baseline

In [ ]:
import json, pandas as pd
base = json.load(open(ROOT / 'models/metrics.json'))
temp = json.load(open(ROOT / 'models/temporal_metrics.json'))
rows = []
for ex in base['exercises']:
    rows.append({
        'exercise': ex,
        'rf_video_f1': base['exercises'][ex]['test']['video']['f1'],
        'rf_video_acc': base['exercises'][ex]['test']['video']['accuracy'],
        'rf_false_accept': base['exercises'][ex]['test']['video']['false_accept_rate_bad_as_good'],
        'seq_video_f1': temp.get(ex, {}).get('test', {}).get('video', {}).get('f1'),
        'seq_video_acc': temp.get(ex, {}).get('test', {}).get('video', {}).get('accuracy'),
        'seq_false_accept': temp.get(ex, {}).get('test', {}).get('video', {}).get('false_accept_rate_bad_as_good'),
    })
print(pd.DataFrame(rows).to_string(index=False))

The temporal model only ships if it wins on the **test** split. With 26
participants a 1-3% difference is noise, so prefer the simpler model unless the
margin is clearly larger.